# Querdenk-Modell – kompletter Nachtlauf
**Stufe 1** (Denkprozess lernen, max. 3 h) → **Stufe 2** (selbst erfinden per GRPO) bis kurz vor dem 12-h-Limit von Kaggle. Danach wird automatisch gespeichert.

**Start:** GPU T4 x2 · Internet an · **Save Version → Save & Run All (Commit)** → Rechner darf aus sein.

**Weitertrainieren:** Neuen Lauf starten und vorher den Output dieses Laufs über **Add Input → Your Work** hinzufügen. Das Notebook erkennt die Checkpoints, überspringt Stufe 1 und setzt Stufe 2 fort.

In [ ]:
import time; START = time.time()   # Uhr für das Zeitbudget läuft ab hier
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if os.path.exists("repo"):
    subprocess.run(["git", "-C", "repo", "pull", "-q"], check=True)   # immer neuester Stand
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
# === Zeitbudget: Kaggle bricht nach 12 h hart ab – wir stoppen vorher sauber ===
from transformers import TrainerCallback
class Zeitlimit(TrainerCallback):
    def __init__(self, deadline, name): self.deadline, self.name = deadline, name
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"⏱ {self.name}: Zeitbudget erreicht bei Schritt {state.global_step} – speichere und stoppe.")
            control.should_training_stop = True; control.should_save = True
        return control

RUN = CFG["run"]
DEADLINE_ALL = START + RUN["time_budget_hours"] * 3600 - 25 * 60   # 25 Min Puffer für Speichern + Abschlusstest
DEADLINE_SFT = min(DEADLINE_ALL, START + RUN["sft_max_hours"] * 3600)
def rest(): return f"{(DEADLINE_ALL - time.time()) / 3600:.1f} h übrig"
print("Budget gesamt:", rest())

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json, re, torch

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: eingebauten Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

def generate(model, tokenizer, question, max_new_tokens=600, temperature=0.8):
    p = chat(tokenizer, [{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def response_parts(model_name):
    if "qwen" in model_name.lower():
        return "<|im_start|>user\n", "<|im_start|>assistant\n"
    return "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
import glob, shutil, os
from unsloth import PatchFastRL
try:
    PatchFastRL("GRPO", FastLanguageModel)
except Exception:
    pass

# Fortsetzen? Output eines früheren Laufs als Input hinzufügen (Add Input → Your Work)
step_of = lambda p: int(p.rstrip("/").split("-")[-1])
prev_grpo = sorted(glob.glob("/kaggle/input/**/checkpoints_grpo/checkpoint-*", recursive=True), key=step_of)
prev_sft  = glob.glob("/kaggle/input/**/adapter_sft", recursive=True)
if prev_grpo:
    shutil.copytree(os.path.dirname(prev_grpo[-1]), "checkpoints_grpo", dirs_exist_ok=True)
    LOAD_FROM, MODE = prev_grpo[-1], "GRPO fortsetzen"
elif prev_sft:
    LOAD_FROM, MODE = prev_sft[0], "nur GRPO (SFT schon fertig)"
else:
    LOAD_FROM, MODE = CFG["base_model"], "komplett (SFT + GRPO)"
print("Modus:", MODE, "| lade:", LOAD_FROM)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LOAD_FROM, max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
if MODE.startswith("komplett"):
    L = CFG["lora"]
    model = FastLanguageModel.get_peft_model(
        model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42)
else:
    for n, p in model.named_parameters():
        if "lora_" in n:
            p.requires_grad = True
model.print_trainable_parameters()

In [ ]:
# === Stufe 1: Denkprozess lernen ===
if MODE.startswith("komplett"):
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only
    to_text = lambda r: {"text": chat(tokenizer, r["messages"])}
    train = load_dataset("json", data_files="repo/data/sft.jsonl", split="train").map(to_text)
    val   = load_dataset("json", data_files="repo/data/sft_val.jsonl", split="train").map(to_text)
    S = CFG["sft"]
    steps_per_epoch = max(1, len(train) // (S["batch_size"] * S["grad_accum"]))
    sft = SFTTrainer(
        model=model, processing_class=tokenizer, train_dataset=train, eval_dataset=val,
        callbacks=[Zeitlimit(DEADLINE_SFT, "SFT")],
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=CFG["max_seq_length"],
            per_device_train_batch_size=S["batch_size"], gradient_accumulation_steps=S["grad_accum"],
            per_device_eval_batch_size=1, num_train_epochs=S["epochs"], learning_rate=S["lr"],
            warmup_steps=max(1, int(0.05 * steps_per_epoch * S["epochs"])),
            lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=5,
            eval_strategy="steps", eval_steps=S["eval_steps"],
            save_strategy="steps", save_steps=S["eval_steps"], save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
            output_dir="checkpoints_sft", report_to="none", seed=42))
    sft = train_on_responses_only(sft, *response_parts(CFG["base_model"]))
    sft.train()
    import pandas as pd
    print(pd.DataFrame(sft.state.log_history)[["step","eval_loss"]].dropna().to_string(index=False))
    print("Bestes Checkpoint:", sft.state.best_model_checkpoint)
    model.save_pretrained("adapter_sft"); tokenizer.save_pretrained("adapter_sft")
    shutil.rmtree("checkpoints_sft", ignore_errors=True)   # Platz sparen, adapter_sft reicht
    del sft; torch.cuda.empty_cache()
else:
    print("SFT übersprungen –", MODE)
print(rest())

In [ ]:
# Zwischenstand nach Stufe 1: Business-Frage (nie trainiert), Basis vs. trainiert
FastLanguageModel.for_inference(model)
qs = [json.loads(open("repo/eval/business_holdout.jsonl").readline())["prompt"],
      json.loads(open("repo/eval/business_evolution.jsonl").readline())["prompt"]]
for q in qs:
    print("FRAGE:", q)
    with model.disable_adapter():
        print("\n--- BASISMODELL ---\n", generate(model, tokenizer, q)[:1500])
    print("\n--- NACH STUFE 1 ---\n", generate(model, tokenizer, q)[:1500])

## Stufe 2: Judge, Belohnung, GRPO

In [ ]:
G = CFG["grpo"]
FastLanguageModel.for_inference(model)  # bleibt so bis zum GRPO-Start
# Judge + Embeddings auf GPU 1
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
JDEV = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
j_tok = AutoTokenizer.from_pretrained(G["judge_model"]); j_tok.padding_side = "left"
judge = AutoModelForCausalLM.from_pretrained(G["judge_model"], device_map={"": JDEV}, torch_dtype=torch.float16).eval()
embedder = SentenceTransformer(G["embed_model"], device=JDEV)

JUDGE_PROMPT = """Problem: {q}

Vorgeschlagene Idee: {idea}

Bewerte NUR, ob diese Idee das Problem tatsächlich lösen kann und realistisch umsetzbar, sicher und legal ist.
Originalität spielt keine Rolle. Vage Floskeln ohne konkrete Umsetzung bekommen höchstens 3.
Antworte ausschließlich mit einer ganzen Zahl von 0 bis 10."""

@torch.no_grad()
def judge_scores(questions, ideas, bs=8):
    out = []
    for i in range(0, len(ideas), bs):
        texts = [chat(j_tok, [{"role":"user","content":JUDGE_PROMPT.format(q=q, idea=d[:1200])}], add_generation_prompt=True)
                 for q, d in zip(questions[i:i+bs], ideas[i:i+bs])]
        enc = j_tok(texts, return_tensors="pt", padding=True).to(JDEV)
        gen = judge.generate(**enc, max_new_tokens=4, do_sample=False)
        for g in gen[:, enc.input_ids.shape[1]:]:
            m = re.search(r"\d+", j_tok.decode(g, skip_special_tokens=True))
            out.append(min(10, int(m.group())) / 10 if m else 0.0)
    return out

def emb(texts):
    return embedder.encode([f"query: {t}" for t in texts], normalize_embeddings=True, convert_to_tensor=True)

print(judge_scores(["Wie hält man Brot länger frisch?"], ["Brot in ein Leinentuch wickeln und in einem Brotkasten lagern."]))

In [ ]:
# Belohnungsfunktionen
def best_idea(text):
    m = re.search(r"^Beste Idee:\s*(.+)", text, re.M | re.S)
    return (m.group(1) if m else text[-600:]).strip()

def question_of(prompt):
    m = re.search(r"user\n(.*?)<\|im_end\|>", prompt, re.S)
    return m.group(1).strip() if m else prompt[-400:]

def r_format(prompts, completions, **kw):
    out = []
    for c in completions:
        L = c.split("\n")
        tools = sum(l.startswith("Werkzeug – ") and "→ Rohidee:" in l for l in L)
        s = 0.25 * any(l.startswith("Annahmen:") for l in L)
        s += 0.25 * (3 <= tools <= 4)
        s += 0.2 * any(l.startswith("Aussortiert:") for l in L)
        s += 0.3 * any(l.startswith("Beste Idee:") for l in L)
        out.append(s)
    return out

LOG = []
def r_creative_valid(prompts, completions, obvious, **kw):
    qs = [question_of(p) for p in prompts]
    ideas = [best_idea(c) for c in completions]
    valid = judge_scores(qs, ideas)
    E_i, E_o = emb(ideas), emb([o or q for o, q in zip(obvious, qs)])
    novelty = (1 - (E_i * E_o).sum(-1)).clamp(min=0).mul(4).clamp(max=1).tolist()
    LOG.append({"valid": sum(valid)/len(valid), "novelty": sum(novelty)/len(novelty)})
    return [v * (0.4 + 0.6 * n) for v, n in zip(valid, novelty)]   # ohne Gültigkeit keine Punkte

def r_diversity(prompts, completions, **kw):
    E = emb([best_idea(c) for c in completions]); out = []
    for i, p in enumerate(prompts):
        others = [j for j, q in enumerate(prompts) if q == p and j != i]
        out.append(0.0 if not others else float((1 - (E[others] @ E[i])).mean().mul(4).clamp(0, 1)))
    return out

def r_length(prompts, completions, **kw):
    return [0.0 if len(c.split()) <= 380 else -min(1.0, (len(c.split()) - 380) / 200) for c in completions]

W = G["reward_weights"]
REWARDS = [r_format, r_creative_valid, r_diversity, r_length]
WEIGHTS = [W["format"], W["creative_valid"], W["diversity"], W["length"]]

In [ ]:
from datasets import load_dataset
G = CFG["grpo"]
ds = load_dataset("json", data_files="repo/data/grpo_prompts.jsonl", split="train")
ds = ds.map(lambda r: {"prompt": chat(tokenizer, [{"role":"user","content":r["prompt"]}], add_generation_prompt=True),
                       "obvious": r["obvious"]}).shuffle(seed=42)
print(len(ds), "Fragen |", rest())

In [ ]:
# === Stufe 2: selbst erfinden – läuft bis kurz vor Ende des Zeitbudgets ===
from trl import GRPOTrainer, GRPOConfig
FastLanguageModel.for_training(model)
args = GRPOConfig(
    learning_rate=G["lr"], max_steps=G["max_steps"], save_steps=G["save_steps"], save_total_limit=3,
    per_device_train_batch_size=G["batch_size"], gradient_accumulation_steps=G["grad_accum"],
    num_generations=G["num_generations"], temperature=G["temperature"],
    max_prompt_length=G["max_prompt_length"], max_completion_length=G["max_completion_length"],
    reward_weights=WEIGHTS, optim="adamw_8bit", lr_scheduler_type="constant_with_warmup", warmup_steps=5,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    logging_steps=1, output_dir="checkpoints_grpo", report_to="none", seed=42)
grpo = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=REWARDS, args=args,
                   train_dataset=ds, callbacks=[Zeitlimit(DEADLINE_ALL, "GRPO")])
last = sorted(glob.glob("checkpoints_grpo/checkpoint-*"), key=step_of)
grpo.train(resume_from_checkpoint=last[-1] if last else None)
model.save_pretrained("adapter_grpo"); tokenizer.save_pretrained("adapter_grpo")
print("Gespeichert: adapter_grpo |", rest())

In [ ]:
import pandas as pd
df = pd.DataFrame(LOG)
if len(df):
    k = max(1, len(df) // 10)
    print("Gültigkeit & Originalität im Verlauf (gleitender Schnitt):")
    print(df.rolling(k).mean().iloc[::k].round(3).to_string())

FastLanguageModel.for_inference(model)
qs = [json.loads(open("repo/eval/business_holdout.jsonl").readline())["prompt"],
      json.loads(open("repo/eval/business_evolution.jsonl").readline())["prompt"]]
for q in qs:
    print("\nFRAGE (Business, nie trainiert):", q)
    print("\n--- NACH STUFE 2 ---\n", generate(model, tokenizer, q))
print("\nFertig. Output enthält adapter_sft, adapter_grpo und checkpoints_grpo (zum Fortsetzen).")